In [ ]:

# Run this notebook with a SageMath kernel.
import csv
import json
import random
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from sage.version import version as SAGE_VERSION

# ---------------------------
# Dataset configuration
# ---------------------------

OUTPUT_DIR = Path("generated_datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set to True first to validate the environment and CSV parser quickly.
SMOKE_TEST = True

# For the full experiment, set SMOKE_TEST = False.
# The generator uses the largest prime p < 2^bits for every dataset.
DATASET_SPECS = [
    {"field_bits": 25, "instances": 100_000, "seed": 2026070501},
    {"field_bits": 26, "instances": 100_000, "seed": 2026070502},
    {"field_bits": 27, "instances": 100_000, "seed": 2026070503},
    {"field_bits": 28, "instances": 100_000, "seed": 2026070504},
    {"field_bits": 29, "instances": 100_000, "seed": 2026070505},
    {"field_bits": 30, "instances": 100_000, "seed": 2026070506},
]

if SMOKE_TEST:
    DATASET_SPECS = [
        {"field_bits": 25, "instances": 100, "seed": 2026070501},
    ]

# The following constraints define which composite target subgroups are accepted.
# Keep them unchanged across all datasets and record them in the manuscript/manifest.
REQUIRE_AT_LEAST_TWO_DISTINCT_PRIMES = True
MIN_SUBGROUP_ORDER_BITS_OFFSET = 3   # Require N to have at least field_bits - 3 bits.
MIN_LARGEST_PRIME_FACTOR_BITS = 8    # Avoid trivial digit-level PH subproblems.
MAX_ATTEMPTS_PER_DATASET = 20_000_000

# Existing C# CSV parser expects exactly these 11 fields and no header.
WRITE_HEADER = False
OVERWRITE_EXISTING = False


In [ ]:

CSV_COLUMNS = [
    "p",
    "a",
    "b",
    "curve_order",
    "P_x",
    "P_y",
    "P_order",
    "Q_x",
    "Q_y",
    "Q_order",
    "expected_k",
]


def largest_prime_below_power_of_two(field_bits):
    """Return the largest prime p strictly below 2^field_bits."""
    candidate = Integer(2)**Integer(field_bits) - 1
    while not is_prime(candidate):
        candidate -= 1
    return Integer(candidate)


def factor_pairs(n):
    """Return the exact prime-power factorisation of n as Python-safe integer pairs."""
    n = Integer(n)
    pairs = [(Integer(prime), Integer(exponent)) for prime, exponent in factor(n)]
    reconstructed = prod(prime**exponent for prime, exponent in pairs)
    if reconstructed != n:
        raise RuntimeError(f"Factorisation check failed for N={n}.")
    return pairs


def factorisation_text(pairs):
    return " * ".join(f"{int(prime)}^{int(exponent)}" for prime, exponent in pairs)


def order_is_acceptable(N, field_bits):
    """
    Decide whether N is a suitable composite target-subgroup order.

    The default policy ensures:
    - N is composite;
    - N is large enough relative to the field;
    - CRT is exercised through at least two distinct primes;
    - digit-level PH subproblems are not all trivial.
    """
    N = Integer(N)
    if N <= 3 or is_prime(N):
        return False, None

    pairs = factor_pairs(N)

    if REQUIRE_AT_LEAST_TWO_DISTINCT_PRIMES and len(pairs) < 2:
        return False, pairs

    if N.nbits() < max(2, Integer(field_bits) - MIN_SUBGROUP_ORDER_BITS_OFFSET):
        return False, pairs

    largest_prime_bits = max(prime.nbits() for prime, _ in pairs)
    if largest_prime_bits < MIN_LARGEST_PRIME_FACTOR_BITS:
        return False, pairs

    return True, pairs


def finite_xy(point):
    """Return integer affine coordinates and reject the point at infinity."""
    if point.is_zero():
        raise ValueError("The point at infinity cannot be written in the CSV affine-coordinate schema.")
    x_coord, y_coord = point.xy()
    return Integer(x_coord), Integer(y_coord)


def validate_instance(E, p, a, b, curve_order, P, N, Q, Q_order, k, factors):
    """Perform all consistency checks before a row is written."""
    if 4 * a**3 + 27 * b**2 == 0:
        raise AssertionError("Singular short-Weierstrass curve generated.")

    if not E.is_on_curve(*P.xy()) or not E.is_on_curve(*Q.xy()):
        raise AssertionError("A stored point is not on the generated curve.")

    if P.is_zero() or Q.is_zero():
        raise AssertionError("The CSV schema does not support the point at infinity.")

    if Integer(P.order()) != Integer(N):
        raise AssertionError("Stored order of P is incorrect.")

    expected_q_order = Integer(N) // gcd(Integer(N), Integer(k))
    if Integer(Q.order()) != expected_q_order or Integer(Q.order()) != Integer(Q_order):
        raise AssertionError("Stored order of Q is incorrect.")

    if Q != Integer(k) * P:
        raise AssertionError("The DLP relation Q = kP is invalid.")

    if not (1 <= Integer(k) < Integer(N)):
        raise AssertionError("k must be in [1, N-1].")

    if is_prime(Integer(N)):
        raise AssertionError("The target-subgroup order must be composite.")

    if prod(prime**exponent for prime, exponent in factors) != Integer(N):
        raise AssertionError("The saved factorisation does not reconstruct N.")

    if Integer(p) != Integer(E.base_field().order()):
        raise AssertionError("The curve field does not match p.")

    if Integer(curve_order) != Integer(E.cardinality()):
        raise AssertionError("The stored curve order is incorrect.")


def make_unique_key(a, b, P, k):
    """A unique ECDLP instance key for one fixed field p."""
    px, py = finite_xy(P)
    return (int(a), int(b), int(px), int(py), int(k))


In [ ]:

def generate_dataset(field_bits, instances, seed, output_dir=OUTPUT_DIR):
    """
    Generate one CSV dataset and two companion files:
    - <name>.csv: exactly the 11 columns expected by the current C# parser;
    - <name>.audit.csv: per-row target-order factorisation for verification;
    - <name>.manifest.json: generation parameters and aggregate statistics.

    One accepted ECDLP instance is emitted per accepted curve, making curve parameters unique
    within a dataset. Set a different policy if multiple instances per curve are desired.
    """
    field_bits = Integer(field_bits)
    instances = Integer(instances)
    p = largest_prime_below_power_of_two(field_bits)
    F = GF(p)

    dataset_name = f"ecdpl_composite_ph_{int(field_bits)}bit_{int(instances)}"
    csv_path = Path(output_dir) / f"{dataset_name}.csv"
    audit_path = Path(output_dir) / f"{dataset_name}.audit.csv"
    manifest_path = Path(output_dir) / f"{dataset_name}.manifest.json"

    for path in (csv_path, audit_path, manifest_path):
        if path.exists() and not OVERWRITE_EXISTING:
            raise FileExistsError(
                f"{path} already exists. Set OVERWRITE_EXISTING = True or choose another output directory."
            )

    # Seed both SageMath's random source and Python's local random source.
    set_random_seed(Integer(seed))
    rng = random.Random(int(seed))

    used_curves = set()
    used_instances = set()
    factorisation_counter = Counter()
    order_bit_lengths = []
    q_order_bit_lengths = []

    attempts = 0
    written = 0

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file, \
         audit_path.open("w", newline="", encoding="utf-8") as audit_file:

        writer = csv.writer(csv_file)
        audit_writer = csv.writer(audit_file)

        if WRITE_HEADER:
            writer.writerow(CSV_COLUMNS)

        audit_writer.writerow([
            "row_index",
            "P_order",
            "P_order_factorisation",
            "largest_prime_factor",
            "largest_prime_factor_bits",
            "Q_order",
        ])

        while written < instances:
            attempts += 1
            if attempts > MAX_ATTEMPTS_PER_DATASET:
                raise RuntimeError(
                    f"Stopped after {attempts} attempts with only {written}/{instances} accepted rows. "
                    "Relax the subgroup-order policy or inspect the acceptance criteria."
                )

            a = F.random_element()
            b = F.random_element()

            # Explicitly reject singular curves before constructing E.
            if 4 * a**3 + 27 * b**2 == 0:
                continue

            curve_key = (int(a), int(b))
            if curve_key in used_curves:
                continue

            E = EllipticCurve(F, [a, b])
            curve_order = Integer(E.cardinality())

            P = E.random_point()
            if P.is_zero():
                continue

            N = Integer(P.order())
            accepted, factors = order_is_acceptable(N, field_bits)
            if not accepted:
                continue

            # The scalar is nonzero and strictly below N, so Q is not the point at infinity.
            k = Integer(rng.randrange(1, int(N)))
            Q = Integer(k) * P
            if Q.is_zero():
                continue

            Q_order = Integer(Q.order())

            instance_key = make_unique_key(a, b, P, k)
            if instance_key in used_instances:
                continue

            validate_instance(
                E=E,
                p=p,
                a=a,
                b=b,
                curve_order=curve_order,
                P=P,
                N=N,
                Q=Q,
                Q_order=Q_order,
                k=k,
                factors=factors,
            )

            px, py = finite_xy(P)
            qx, qy = finite_xy(Q)

            # This is the 11-column schema used by the existing C# parser.
            row = [
                int(p),
                int(a),
                int(b),
                int(curve_order),
                int(px),
                int(py),
                int(N),
                int(qx),
                int(qy),
                int(Q_order),
                int(k),
            ]
            writer.writerow(row)

            largest_prime = max(prime for prime, _ in factors)
            factorisation_counter[factorisation_text(factors)] += 1
            audit_writer.writerow([
                written,
                int(N),
                factorisation_text(factors),
                int(largest_prime),
                int(largest_prime.nbits()),
                int(Q_order),
            ])

            used_curves.add(curve_key)
            used_instances.add(instance_key)
            order_bit_lengths.append(int(N.nbits()))
            q_order_bit_lengths.append(int(Q_order.nbits()))
            written += 1

            if written % 1000 == 0 or written == instances:
                print(
                    f"{dataset_name}: {written}/{instances} rows "
                    f"(attempts={attempts}, acceptance={written / attempts:.2%})"
                )

    manifest = {
        "generator": "sagemath_composite_order_dataset_generator",
        "generator_version": "1.0",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "csv_schema": CSV_COLUMNS,
        "csv_has_header": WRITE_HEADER,
        "field_bits": int(field_bits),
        "p": int(p),
        "instances": int(instances),
        "seed": int(seed),
        "attempts": attempts,
        "acceptance_rate": written / attempts,
        "sagemath_version": SAGE_VERSION,
        "python_version": sys.version,
        "subgroup_policy": {
            "target_order": "N = ord(P)",
            "require_composite_order": True,
            "require_at_least_two_distinct_primes": REQUIRE_AT_LEAST_TWO_DISTINCT_PRIMES,
            "minimum_target_order_bits": max(2, int(field_bits) - MIN_SUBGROUP_ORDER_BITS_OFFSET),
            "minimum_largest_prime_factor_bits": MIN_LARGEST_PRIME_FACTOR_BITS,
        },
        "target_order_bit_length": {
            "minimum": min(order_bit_lengths),
            "maximum": max(order_bit_lengths),
            "mean": sum(order_bit_lengths) / len(order_bit_lengths),
        },
        "Q_order_bit_length": {
            "minimum": min(q_order_bit_lengths),
            "maximum": max(q_order_bit_lengths),
            "mean": sum(q_order_bit_lengths) / len(q_order_bit_lengths),
        },
        "factorisation_pattern_counts": dict(sorted(factorisation_counter.items())),
        "files": {
            "dataset_csv": csv_path.name,
            "audit_csv": audit_path.name,
        },
    }

    with manifest_path.open("w", encoding="utf-8") as manifest_file:
        json.dump(manifest, manifest_file, indent=2, ensure_ascii=False)

    print(f"\nCreated: {csv_path}")
    print(f"Audit file: {audit_path}")
    print(f"Manifest: {manifest_path}")
    return csv_path, audit_path, manifest_path


In [ ]:

def validate_written_csv(csv_path, expected_rows, has_header=WRITE_HEADER):
    """
    Re-read the generated CSV and validate its format and mathematical relations.
    This checks the final file, not merely the in-memory values before writing.
    """
    csv_path = Path(csv_path)
    checked = 0

    with csv_path.open("r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)

        if has_header:
            header = next(reader)
            if header != CSV_COLUMNS:
                raise AssertionError("Unexpected CSV header.")

        for row_index, row in enumerate(reader):
            if len(row) != len(CSV_COLUMNS):
                raise AssertionError(
                    f"Row {row_index} has {len(row)} columns; expected {len(CSV_COLUMNS)}."
                )

            (
                p, a, b, curve_order,
                px, py, p_order,
                qx, qy, q_order,
                k,
            ) = map(Integer, row)

            F = GF(p)
            E = EllipticCurve(F, [F(a), F(b)])
            P = E(F(px), F(py))
            Q = E(F(qx), F(qy))
            factors = factor_pairs(p_order)

            validate_instance(
                E=E,
                p=p,
                a=F(a),
                b=F(b),
                curve_order=curve_order,
                P=P,
                N=p_order,
                Q=Q,
                Q_order=q_order,
                k=k,
                factors=factors,
            )
            checked += 1

    if checked != expected_rows:
        raise AssertionError(f"Expected {expected_rows} rows but validated {checked}.")

    print(f"Validation passed: {checked} rows in {csv_path.name}.")


## Generate the datasets

Run the next cell with `SMOKE_TEST = True` first. After the audit and CSV format look correct, set `SMOKE_TEST = False`, choose an empty output directory, and run it again for the six full datasets. The script does not write factorisation values into the main 11-column CSV, preserving compatibility with the existing parser; those details are stored in the audit file and manifest.


In [ ]:

generated_files = []

for spec in DATASET_SPECS:
    csv_path, audit_path, manifest_path = generate_dataset(
        field_bits=spec["field_bits"],
        instances=spec["instances"],
        seed=spec["seed"],
    )
    validate_written_csv(csv_path, expected_rows=spec["instances"])
    generated_files.append((csv_path, audit_path, manifest_path))

generated_files
